<a href="https://colab.research.google.com/github/Prajnagowda10/Cloth-Size-Prediction-CNN/blob/main/miniproject2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q gradio huggingface_hub torch torchvision pillow

import torch
import torch.nn as nn
import gradio as gr

from torchvision import models, transforms
from huggingface_hub import hf_hub_download
from PIL import Image

REPO_ID = "ilian-hadzhidimitrov/food-classifier-resnet50"

# Download pretrained CNN weights and class labels
weights_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="resnet50_food101_best.pt"
)

classes_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="classes_food101.txt"
)

with open(classes_path, "r") as f:
    class_names = f.read().splitlines()

# Create the ResNet50 CNN architecture
model = models.resnet50(weights=None)
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(2048, 101)
)

state_dict = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state_dict)
model.eval()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
model = model.to(device)

# Preprocess images as required by the model
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

print("Food CNN model loaded successfully!")
print("Number of categories:", len(class_names))

def predict_food(image):
    if image is None:
        return {}

    image = image.convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    top_probs, top_indices = torch.topk(
        probabilities, k=5
    )

    results = {}
    for prob, idx in zip(
        top_probs.cpu().tolist(),
        top_indices.cpu().tolist()
    ):
        results[class_names[idx]] = float(prob)

    return results


with gr.Blocks(title="Food Image Classification CNN") as demo:
    gr.Markdown(
        """
        # 🍽️ Food Image Classification Using CNN
        Upload a food image or capture one with your webcam.
        The ResNet50 CNN will predict the most likely food categories.
        """
    )

    image_input = gr.Image(
        label="Upload or capture a food image",
        type="pil",
        sources=["upload", "webcam"]
    )

    predict_button = gr.Button("Predict Food", variant="primary")

    output = gr.Label(
        label="Top 5 food predictions",
        num_top_classes=5
    )

    predict_button.click(
        fn=predict_food,
        inputs=image_input,
        outputs=output
    )

demo.launch(share=True)



Food CNN model loaded successfully!
Number of categories: 101
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/29 15:08:39 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

In [9]:

!pip install -q gradio torch torchvision huggingface_hub

import torch
import torch.nn as nn
from torchvision import models, transforms
from huggingface_hub import hf_hub_download
from PIL import Image
import gradio as gr

# Download individual model files
REPO_ID = "nateraw/food"

weights_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="pytorch_model.bin"
)

print("Model file downloaded:", weights_path)

!pip install -q transformers

from transformers import pipeline

classifier = pipeline(
    "image-classification",
    model="nateraw/food"
)

print("Food image classifier loaded successfully!")

def predict_food(image):
    if image is None:
        return "Please upload a food image."

    image = image.convert("RGB")
    predictions = classifier(image, top_k=5)

    result = {}
    for item in predictions:
        result[item["label"]] = float(item["score"])

    return result


demo = gr.Interface(
    fn=predict_food,
    inputs=gr.Image(
        type="pil",
        sources=["upload", "webcam"],
        label="Upload food image or take a photo"
    ),
    outputs=gr.Label(
        num_top_classes=5,
        label="Predicted food"
    ),
    title="Food Image Classification Using CNN",
    description="Upload a food image or use your webcam to predict its category."
)

demo.launch(share=True)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  344MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Model file downloaded: /root/.cache/huggingface/hub/models--nateraw--food/snapshots/8991abd49ea01ebf502aeda51d4f12a59c603e01/pytorch_model.bin


config.json:   0%|          | 0.00/5.58k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

Food image classifier loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/29 15:05:42 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

In [ ]:

!pip install -q gradio transformers torch torchvision pillow matplotlib

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import torch

from PIL import Image, ImageOps
from transformers import pipeline

PROJECT_NAME = "Food Image Classification Using CNN"

DEVICE = 0 if torch.cuda.is_available() else -1

print("Project:", PROJECT_NAME)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Device:", "GPU" if DEVICE == 0 else "CPU")

MODEL_NAME = "nateraw/food"

try:
    classifier = pipeline(
        task="image-classification",
        model=MODEL_NAME,
        device=DEVICE
    )
    print("Model loaded successfully!")
except Exception as error:
    print("Model loading failed:")
    print(str(error))
    raise RuntimeError(
        "Check the model ID and internet connection, "
        "then retry with a compatible Food-101 checkpoint."
    )

def preprocess_image(image):
    """
    Prepare an uploaded image for food classification.
    """
    if image is None:
        raise ValueError("Please upload an image.")

    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.array(image))

    image = ImageOps.exif_transpose(image)
    image = image.convert("RGB")

    # Keep excessively large images manageable
    image.thumbnail((1200, 1200))

    return image

def predict_food(image, top_k):
    """
    Predict the food category and return the top-k scores.
    """
    if image is None:
        return {}, "Please upload a food image."

    try:
        processed_image = preprocess_image(image)

        predictions = classifier(
            processed_image,
            top_k=int(top_k)
        )

        results = {}

        for prediction in predictions:
            label = prediction["label"]
            score = float(prediction["score"])
            results[label] = score

        if results:
            best_label = max(results, key=results.get)
            best_score = results[best_label] * 100

            message = (
                f"Predicted food: {best_label}\n"
                f"Confidence: {best_score:.2f}%"
            )
        else:
            message = "No prediction was returned."

        return results, message

    except Exception as error:
        return {}, f"Prediction failed: {error}"

def classify_uploaded_image(image, top_k):
    predictions, message = predict_food(image, top_k)
    return predictions, message


with gr.Blocks(
    title=PROJECT_NAME,
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # 🍽️ Food Image Classification Using CNN

        Upload a food image or capture a photo using your webcam.

        The deep learning model analyzes the image and returns
        the most likely food categories with confidence scores.
        """
    )

    with gr.Row():

        with gr.Column():
            image_input = gr.Image(
                label="Food Image",
                type="pil",
                sources=["upload", "webcam"]
            )

            top_k_input = gr.Slider(
                minimum=1,
                maximum=5,
                value=5,
                step=1,
                label="Number of predictions"
            )

            predict_button = gr.Button(
                "Classify Food",
                variant="primary"
            )

            clear_button = gr.Button("Clear")

        with gr.Column():
            prediction_output = gr.Label(
                label="Food Categories",
                num_top_classes=5
            )

            message_output = gr.Textbox(
                label="Prediction Details",
                lines=3
            )

    predict_button.click(
        fn=classify_uploaded_image,
        inputs=[image_input, top_k_input],
        outputs=[prediction_output, message_output]
    )

    clear_button.click(
        fn=lambda: (None, {}, ""),
        inputs=[],
        outputs=[
            image_input,
            prediction_output,
            message_output
        ]
    )

    gr.Markdown(
        """
        ### How to use
        1. Upload an image or select the webcam option.
        2. Choose how many predictions to display.
        3. Click **Classify Food**.
        4. Read the predicted categories and confidence scores.

        Note: Predictions depend on the categories represented
        in the selected model's training data.
        """
    )

print("Web interface created.")

demo.launch(
    share=True,
    debug=True
)

import csv
from datetime import datetime

CSV_FILE = "food_prediction_history.csv"


def save_prediction(food_name, confidence):
    file_exists = os.path.exists(CSV_FILE)

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow([
                "Timestamp",
                "Predicted Food",
                "Confidence"
            ])

        writer.writerow([
            datetime.now().isoformat(timespec="seconds"),
            food_name,
            round(float(confidence), 4)
        ])

    print("Prediction saved to", CSV_FILE)


Project: Food Image Classification Using CNN
PyTorch version: 2.11.0+cpu
GPU available: False
Device: CPU


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model loaded successfully!


/tmp/ipykernel_564/1578446718.py:101: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Web interface created.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/29 15:12:44 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>